# RAG over Astronomy Articles — Baseline + Improved Pipeline

**Course:** AI09/AI10 SEM 02 — Generative AI Applications (Assignment 01)
**Dataset:** 35 curated Wikipedia astronomy articles (auto-scraped)
**Stack:** sentence-transformers + ChromaDB + BM25 + Cross-encoder + Phi-3-mini (open LLM, runs on Kaggle T4)
**Author:** Salma Areef Syed

> **Setup on Kaggle:** Settings → Accelerator → **GPU T4 x2** (or P100). Internet → **On**. Then run all cells top-to-bottom.

## 1. Install dependencies

In [1]:
!pip install -q wikipedia sentence-transformers chromadb transformers accelerate bitsandbytes rank_bm25 pandas openpyxl -U

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.9/588.9 kB 14.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 75.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 106.3 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 110.9 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.2/668.2 kB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 109.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━

## 2. Imports

In [2]:
import os, re, json, time, warnings
import wikipedia
import torch
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer, CrossEncoder
import chromadb
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from rank_bm25 import BM25Okapi

warnings.filterwarnings('ignore')
print('Torch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

Torch: 2.10.0+cu128 | CUDA available: True
GPU: Tesla T4


## 3. Dataset — scrape 35 curated Wikipedia articles

These 35 articles are intentionally picked to cover four question categories well:
- factually dense (planets, missions) -> direct factual Q's
- topical overlap (Curiosity vs Perseverance, Voyager vs New Horizons) -> retrieval ambiguity
- cosmology + phenomena -> multi-step reasoning

In [3]:
ARTICLES = [
    # Planets / dwarf (9)
    "Mercury (planet)", "Venus", "Earth", "Mars", "Jupiter",
    "Saturn", "Uranus", "Neptune", "Pluto",
    # Missions (10)
    "Voyager program", "Hubble Space Telescope", "James Webb Space Telescope",
    "Cassini–Huygens", "New Horizons", "Perseverance (rover)",
    "Curiosity (rover)", "Chandrayaan-3", "Apollo program", "Artemis program",
    # Stars / galaxies / cosmology (8)
    "Sun", "Milky Way", "Andromeda Galaxy", "Black hole", "Neutron star",
    "Supernova", "Big Bang", "Dark matter",
    # Phenomena (8)
    "Exoplanet", "Asteroid", "Comet", "Solar eclipse", "Nebula",
    "Pulsar", "Quasar", "Gravitational wave",
]
print(f'Target: {len(ARTICLES)} articles')

Target: 35 articles


In [4]:
DATA_DIR = '/kaggle/working/dataset'
os.makedirs(DATA_DIR, exist_ok=True)

docs = {}
failed = []
for title in ARTICLES:
    try:
        page = wikipedia.page(title, auto_suggest=False)
        docs[title] = page.content
    except wikipedia.exceptions.DisambiguationError as e:
        try:
            page = wikipedia.page(e.options[0], auto_suggest=False)
            docs[title] = page.content
        except Exception as e2:
            failed.append((title, str(e2)))
    except Exception as e:
        failed.append((title, str(e)))

print(f'Scraped: {len(docs)} | Failed: {len(failed)}')
for t, e in failed:
    print(f'  FAILED: {t} -> {e}')

for title, content in docs.items():
    safe = re.sub(r'[^a-zA-Z0-9_-]', '_', title)
    with open(f'{DATA_DIR}/{safe}.txt', 'w', encoding='utf-8') as f:
        f.write(content)
print(f'Saved to {DATA_DIR}/')

total_chars = sum(len(c) for c in docs.values())
print(f'Total chars: {total_chars:,} | Avg per article: {total_chars // len(docs):,}')

Scraped: 35 | Failed: 0
Saved to /kaggle/working/dataset/
Total chars: 1,751,911 | Avg per article: 50,054


## 4. Chunking — baseline vs improved

- **Baseline:** fixed 500-character chunks, no overlap. Naive and easy to break.
- **Improved:** paragraph-aware chunker targeting ~200 words with 40-word overlap. Preserves semantic boundaries.

In [5]:
def fixed_chunker(text, size=500, overlap=0):
    chunks, i = [], 0
    while i < len(text):
        chunks.append(text[i:i+size])
        i += max(1, size - overlap)
    return chunks

def paragraph_chunker(text, target_words=200, overlap_words=40):
    paragraphs = [p.strip() for p in re.split(r'\n\s*\n', text) if p.strip()]
    chunks, current, current_len = [], [], 0
    for p in paragraphs:
        pw = p.split()
        if current_len + len(pw) > target_words and current:
            chunks.append(' '.join(current))
            tail = (' '.join(current)).split()[-overlap_words:]
            current = [' '.join(tail)]
            current_len = len(tail)
        current.append(p)
        current_len += len(pw)
    if current:
        chunks.append(' '.join(current))
    return [c for c in chunks if len(c.split()) > 10]

baseline_chunks, improved_chunks = [], []
for title, text in docs.items():
    for c in fixed_chunker(text, 500, 0):
        baseline_chunks.append({'text': c, 'source': title})
    for c in paragraph_chunker(text, 200, 40):
        improved_chunks.append({'text': c, 'source': title})

print(f'Baseline chunks: {len(baseline_chunks)}')
print(f'Improved chunks: {len(improved_chunks)}')

Baseline chunks: 3521
Improved chunks: 1523


## 5. Embedding models — MiniLM (baseline) vs BGE (improved)

In [6]:
embed_baseline = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
embed_improved = SentenceTransformer('BAAI/bge-small-en-v1.5')
print('Baseline embed dim:', embed_baseline.get_sentence_embedding_dimension())
print('Improved embed dim:', embed_improved.get_sentence_embedding_dimension())

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Baseline embed dim: 384
Improved embed dim: 384


## 6. Vector DB — ChromaDB collections

In [7]:
chroma_client = chromadb.Client()

def build_collection(name, chunks, model):
    try:
        chroma_client.delete_collection(name)
    except Exception:
        pass
    col = chroma_client.create_collection(name=name, metadata={'hnsw:space': 'cosine'})
    texts = [c['text'] for c in chunks]
    print(f'Embedding {len(texts)} chunks for collection [{name}]...')
    embs = model.encode(texts, batch_size=64, show_progress_bar=True, convert_to_numpy=True).tolist()
    ids = [f'{name}_{i}' for i in range(len(chunks))]
    metas = [{'source': c['source']} for c in chunks]
    col.add(ids=ids, embeddings=embs, documents=texts, metadatas=metas)
    return col

baseline_col = build_collection('baseline', baseline_chunks, embed_baseline)
improved_col = build_collection('improved', improved_chunks, embed_improved)
print('Collections built.')

Embedding 3521 chunks for collection [baseline]...


Batches:   0%|          | 0/56 [00:00<?, ?it/s]

Embedding 1523 chunks for collection [improved]...


Batches:   0%|          | 0/24 [00:00<?, ?it/s]

Collections built.


## 7. BM25 lexical index (for hybrid retrieval in the improved pipeline)

In [8]:
def tokenize(t):
    return re.findall(r'[a-z0-9]+', t.lower())

bm25_corpus = [tokenize(c['text']) for c in improved_chunks]
bm25 = BM25Okapi(bm25_corpus)
print('BM25 ready.')

BM25 ready.


## 8. Cross-encoder reranker (improved pipeline)

In [9]:
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
print('Reranker loaded.')

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Reranker loaded.


## 9. LLM — Qwen2.5-3B-Instruct (open, runs free on Kaggle T4)

In [10]:
MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
llm = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map='auto',
)
gen = pipeline('text-generation', model=llm, tokenizer=tokenizer)

def generate(prompt, max_new_tokens=256):
    messages = [{'role': 'user', 'content': prompt}]
    out = gen(messages, max_new_tokens=max_new_tokens, do_sample=False, return_full_text=False)
    return out[0]['generated_text'].strip()

print(generate('Say hello in one short sentence.', max_new_tokens=30))

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=30) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Hello!


## 10. Prompt templates — baseline vs improved

In [11]:
BASELINE_PROMPT = '''Use the context below to answer the question.

Context:
{context}

Question: {question}

Answer:'''

IMPROVED_PROMPT = '''You are an astronomy expert. Use ONLY the provided context to answer the question.

Rules:
1. If the answer is NOT in the context, respond exactly: "I don't have enough information in the provided context to answer this question."
2. Cite each fact with its source article in square brackets, e.g., [Hubble Space Telescope].
3. Be concise (2-4 sentences) and strictly factual.
4. Do not invent missions, dates, names, or events that are not in the context.

Example:
Context:
[James Webb Space Telescope] The James Webb Space Telescope launched on 25 December 2021 on an Ariane 5 rocket.
Question: When did JWST launch?
Answer: JWST launched on 25 December 2021 [James Webb Space Telescope].

Context:
{context}

Question: {question}

Answer:'''

## 11. RAG pipelines — baseline and improved

In [12]:
def rag_baseline(question, k=3):
    q_emb = embed_baseline.encode([question]).tolist()
    res = baseline_col.query(query_embeddings=q_emb, n_results=k)
    chunks = [
        {'text': d, 'source': m['source'], 'id': i}
        for d, m, i in zip(res['documents'][0], res['metadatas'][0], res['ids'][0])
    ]
    context = '\n\n'.join(f"[{c['source']}] {c['text']}" for c in chunks)
    answer = generate(BASELINE_PROMPT.format(context=context, question=question))
    return answer, chunks

def rag_improved(question, final_k=3, fetch_k=10):
    # 1. Dense retrieval
    q_emb = embed_improved.encode([question]).tolist()
    dense = improved_col.query(query_embeddings=q_emb, n_results=fetch_k)
    dense_hits = [
        {'text': d, 'source': m['source'], 'id': i}
        for d, m, i in zip(dense['documents'][0], dense['metadatas'][0], dense['ids'][0])
    ]
    # 2. BM25 retrieval
    bm25_scores = bm25.get_scores(tokenize(question))
    top_idx = np.argsort(bm25_scores)[::-1][:fetch_k]
    bm25_hits = [
        {'text': improved_chunks[i]['text'], 'source': improved_chunks[i]['source'], 'id': f'improved_{i}'}
        for i in top_idx if bm25_scores[i] > 0
    ]
    # 3. Union dedupe
    seen, merged = set(), []
    for h in dense_hits + bm25_hits:
        if h['id'] not in seen:
            seen.add(h['id'])
            merged.append(h)
    # 4. Cross-encoder rerank
    pairs = [[question, h['text']] for h in merged]
    scores = reranker.predict(pairs)
    ranked = [h for _, h in sorted(zip(scores, merged), key=lambda x: -x[0])[:final_k]]
    # 5. Generate with improved prompt
    context = '\n\n'.join(f"[{c['source']}] {c['text']}" for c in ranked)
    answer = generate(IMPROVED_PROMPT.format(context=context, question=question))
    return answer, ranked

## 12. Quick sanity check

In [13]:
q = 'When was the James Webb Space Telescope launched?'
print('Q:', q)
print('\n--- BASELINE ---')
a, c = rag_baseline(q)
print(a)
print('Sources:', sorted({x["source"] for x in c}))
print('\n--- IMPROVED ---')
a, c = rag_improved(q)
print(a)
print('Sources:', sorted({x["source"] for x in c}))

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: When was the James Webb Space Telescope launched?

--- BASELINE ---


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The James Webb Space Telescope was launched on December 25, 2021. According to the context provided, it was launched on 25 December 2021 on an Ariane 5 rocket from Kourou, French Guiana.
Sources: ['Hubble Space Telescope', 'James Webb Space Telescope']

--- IMPROVED ---
The James Webb Space Telescope was launched on 25 December 2021 [James Webb Space Telescope].
Sources: ['Hubble Space Telescope', 'James Webb Space Telescope']


## 13. Evaluation set — 18 questions across 4 categories

In [14]:
EVAL_QUESTIONS = [
    # Direct factual (6)
    {'q': 'When was the James Webb Space Telescope launched?', 'type': 'factual',
     'expected': 'December 25, 2021'},
    {'q': 'What is the largest moon of Jupiter?', 'type': 'factual',
     'expected': 'Ganymede'},
    {'q': 'Who was the first person to walk on the Moon?', 'type': 'factual',
     'expected': 'Neil Armstrong (Apollo 11)'},
    {'q': 'How many planets did Voyager 2 visit during its flybys?', 'type': 'factual',
     'expected': 'Four: Jupiter, Saturn, Uranus, Neptune'},
    {'q': 'What is the approximate surface temperature of Venus?', 'type': 'factual',
     'expected': 'About 465 deg C / 737 K'},
    {'q': 'When did Chandrayaan-3 land on the Moon?', 'type': 'factual',
     'expected': 'August 23, 2023'},
    # Multi-step reasoning (5)
    {'q': 'Which is larger, Jupiter or Saturn, and roughly by how much?', 'type': 'reasoning',
     'expected': 'Jupiter is larger by mass and diameter'},
    {'q': 'Which Voyager spacecraft has traveled farthest from Earth, and what was its primary scientific contribution?', 'type': 'reasoning',
     'expected': 'Voyager 1; interstellar space / outer planets imaging'},
    {'q': 'Compare the scientific missions of the Curiosity and Perseverance rovers.', 'type': 'reasoning',
     'expected': 'Both Mars rovers; Curiosity studies habitability, Perseverance seeks biosignatures + sample caching'},
    {'q': 'Which planets in our solar system have ring systems, and which planet has the most prominent rings?', 'type': 'reasoning',
     'expected': 'Jupiter, Saturn, Uranus, Neptune all have rings; Saturn most prominent'},
    {'q': 'What is the relationship between supernovae and neutron stars?', 'type': 'reasoning',
     'expected': 'Neutron stars form from core-collapse supernova remnants of massive stars'},
    # Ambiguous (4)
    {'q': 'Tell me about Mars.', 'type': 'ambiguous',
     'expected': 'Could refer to planet or Mars missions/rovers'},
    {'q': 'What is a giant?', 'type': 'ambiguous',
     'expected': 'Could refer to giant star or gas giant planet'},
    {'q': 'What is the most famous discovery?', 'type': 'ambiguous',
     'expected': 'Underspecified - which field?'},
    {'q': 'How big is it?', 'type': 'ambiguous',
     'expected': 'Underspecified - missing subject'},
    # Hallucination-bait (3)
    {'q': 'Who was the first Indian astronaut to walk on the surface of Mars?', 'type': 'hallucination',
     'expected': 'No one has walked on Mars; question is false-premise'},
    {'q': 'What did Voyager 3 discover during its mission?', 'type': 'hallucination',
     'expected': 'Voyager 3 does not exist; only Voyager 1 and 2 were launched'},
    {'q': 'Summarize the successful Pluto landing mission of 2020.', 'type': 'hallucination',
     'expected': 'No Pluto landing has occurred; New Horizons only did a flyby in 2015'},
]
print(f'Total: {len(EVAL_QUESTIONS)} questions')
print('By type:', pd.Series([q["type"] for q in EVAL_QUESTIONS]).value_counts().to_dict())

Total: 18 questions
By type: {'factual': 6, 'reasoning': 5, 'ambiguous': 4, 'hallucination': 3}


## 14. Run full evaluation — baseline vs improved

In [15]:
rows = []
for i, item in enumerate(EVAL_QUESTIONS, 1):
    print(f'[{i}/{len(EVAL_QUESTIONS)}] ({item["type"]}) {item["q"][:70]}...')
    try:
        ans_b, c_b = rag_baseline(item['q'])
    except Exception as e:
        ans_b, c_b = f'ERROR: {e}', []
    try:
        ans_i, c_i = rag_improved(item['q'])
    except Exception as e:
        ans_i, c_i = f'ERROR: {e}', []
    rows.append({
        '#': i,
        'type': item['type'],
        'question': item['q'],
        'expected': item['expected'],
        'baseline_answer': ans_b,
        'baseline_sources': ', '.join(sorted({c['source'] for c in c_b})),
        'improved_answer': ans_i,
        'improved_sources': ', '.join(sorted({c['source'] for c in c_i})),
    })

df = pd.DataFrame(rows)
df.to_csv('/kaggle/working/eval_results.csv', index=False)
df.to_excel('/kaggle/working/eval_results.xlsx', index=False)
print('Saved: /kaggle/working/eval_results.csv and .xlsx')
df

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[1/18] (factual) When was the James Webb Space Telescope launched?...


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[2/18] (factual) What is the largest moon of Jupiter?...


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[18/18] (hallucination) Summarize the successful Pluto landing mission of 2020....


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Saved: /kaggle/working/eval_results.csv and .xlsx


,#,type,question,expected,baseline_answer,baseline_sources,improved_answer,improved_sources
0,1,factual,When was the James Webb Space Telescope launched?,"December 25, 2021",The James Webb Space Telescope was launched on...,"Hubble Space Telescope, James Webb Space Teles...",The James Webb Space Telescope was launched on...,"Hubble Space Telescope, James Webb Space Teles..."
1,2,factual,What is the largest moon of Jupiter?,Ganymede,The largest moon of Jupiter is Ganymede. It is...,Jupiter,The largest moon of Jupiter is Ganymede [Jupit...,Jupiter
2,3,factual,Who was the first person to walk on the Moon?,Neil Armstrong (Apollo 11),The first person to walk on the Moon was Neil ...,Apollo program,The first person to walk on the Moon was Neil ...,Apollo program
3,4,factual,How many planets did Voyager 2 visit during it...,"Four: Jupiter, Saturn, Uranus, Neptune","During its flybys, Voyager 2 visited two plane...","Neptune, New Horizons, Voyager program",Voyager 2 visited two planets during its flyby...,"Neptune, Saturn, Voyager program"
4,5,factual,What is the approximate surface temperature of...,About 465 deg C / 737 K,The approximate surface temperature of Venus i...,Venus,The average surface temperature of Venus is 73...,Venus
5,6,factual,When did Chandrayaan-3 land on the Moon?,"August 23, 2023",Based on the information provided in the conte...,Chandrayaan-3,Chandrayaan-3 landed on the Moon on 23 August ...,Chandrayaan-3
6,7,reasoning,"Which is larger, Jupiter or Saturn, and roughl...",Jupiter is larger by mass and diameter,Based on the information provided in the conte...,"Jupiter, Saturn",Jupiter is larger than Saturn. Jupiter is appr...,"Jupiter, Saturn"
7,8,reasoning,Which Voyager spacecraft has traveled farthest...,Voyager 1; interstellar space / outer planets ...,Based on the information provided in the conte...,"New Horizons, Voyager program",Voyager 1 has traveled farthest from Earth [Vo...,Voyager program
8,9,reasoning,Compare the scientific missions of the Curiosi...,Both Mars rovers; Curiosity studies habitabili...,"Based on the provided context, the scientific ...","Curiosity (rover), Perseverance (rover)",Curiosity and Perseverance rovers have similar...,"Curiosity (rover), Perseverance (rover)"
9,10,reasoning,Which planets in our solar system have ring sy...,"Jupiter, Saturn, Uranus, Neptune all have ring...","Based on the provided context, the planets in ...","Exoplanet, Jupiter, Saturn",Planets in our solar system with ring systems ...,"Exoplanet, Neptune, Saturn"


## 15. Manual scoring template

Open `eval_results.csv` in Excel/Sheets and fill these columns by hand:

| Column | Values |
|---|---|
| `baseline_correct` | Yes / Partial / No |
| `baseline_hallucinated` | Yes / No |
| `improved_correct` | Yes / Partial / No |
| `improved_hallucinated` | Yes / No |
| `failure_reason` | wrong chunk / hallucination / incomplete / prompt ignored / ambiguous |

Then compute summary metrics in the next cell.

In [16]:
# Once you have filled the manual columns, reload and summarize.
# Example:
# scored = pd.read_csv('/kaggle/working/eval_results_scored.csv')
# print('Baseline accuracy:', (scored['baseline_correct'] == 'Yes').mean())
# print('Improved accuracy:', (scored['improved_correct'] == 'Yes').mean())
# print('Baseline halluc rate:', (scored['baseline_hallucinated'] == 'Yes').mean())
# print('Improved halluc rate:', (scored['improved_hallucinated'] == 'Yes').mean())
print('Fill in the scoring columns in eval_results.csv, then run the analysis here.')

Fill in the scoring columns in eval_results.csv, then run the analysis here.


## 16. Package outputs for submission

In [17]:
import shutil
SUB_DIR = '/kaggle/working/submission'
os.makedirs(SUB_DIR, exist_ok=True)
shutil.copytree(DATA_DIR, f'{SUB_DIR}/dataset', dirs_exist_ok=True)
for f in ['eval_results.csv', 'eval_results.xlsx']:
    src = f'/kaggle/working/{f}'
    if os.path.exists(src):
        shutil.copy(src, SUB_DIR)
shutil.make_archive('/kaggle/working/submission', 'zip', SUB_DIR)
print('Created /kaggle/working/submission.zip')
print('Add manually: notebook (.ipynb), report (.docx), architecture diagram, screenshots.')

Created /kaggle/working/submission.zip
Add manually: notebook (.ipynb), report (.docx), architecture diagram, screenshots.


---

## What still needs human work after this notebook runs

1. **Manual scoring** of the eval CSV (Yes/Partial/No, hallucination Y/N, failure reason).
2. **Failure analysis writeup** — pick 4-6 representative failures and explain root cause.
3. **Architecture diagram** — included as `architecture.png` in the submission folder.
4. **Final Word report** — covering all 8 required sections from the assignment brief.
5. **Screenshots** — capture a few example outputs from the notebook for the report.

The notebook is structured so re-running it after tweaking chunk size / embedding model / prompt is trivial — change one constant and rerun the eval cell.

---

# Glass-box RAG Dashboard (Gradio)

This is the **innovation layer** for the assignment. Instead of a plain chat UI, the dashboard makes every step of the RAG pipeline visible:

- **Side-by-side comparison** — baseline RAG vs improved RAG answer the same question simultaneously
- **Retrieval transparency** — see the exact chunks each pipeline retrieved, with source labels
- **Automatic hallucination heuristic** — flags when an answer contains key terms NOT in the retrieved context
- **Citation check** — verifies the improved pipeline actually cites its sources
- **Failure Lab** — one-click "trap" questions (false-premise) that baseline should fail and improved should refuse
- **One-click full eval** — run all 18 questions through both pipelines and watch the table populate

This directly supports the assignment's emphasis on *understanding where the model fails and why*.

## 17. Install Gradio

In [18]:
!pip install -q gradio plotly scikit-learn -U

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.7/19.7 MB 82.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 101.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 93.7 MB/s eta 0:00:00:00:0100:01


## 18. Hallucination & citation heuristics

Two lightweight checkers that surface in the UI:
- **Hallucination heuristic:** extract proper nouns, numbers, and years from the answer. Check whether they appear in the retrieved context. Low coverage → flag.
- **Citation check:** does the answer contain `[Source Name]` markers matching the retrieved chunks?
- **Refusal detect:** did the model say "I don't have enough information"? (Good behavior on false-premise questions.)

In [19]:
import re

def hallucination_heuristic(answer, chunks):
    """Lightweight check: do key terms in the answer appear in the retrieved context?"""
    if not answer or not chunks:
        return {'flag': 'NO_CHECK', 'coverage': 0.0, 'detail': 'No content to check.'}
    if "don't have enough information" in answer.lower() or "do not have enough information" in answer.lower():
        return {'flag': 'REFUSED', 'coverage': 1.0,
                'detail': 'Model refused to answer - correct behavior for false-premise questions.'}
    context_text = ' '.join(c['text'].lower() for c in chunks)
    # Key tokens = capitalized words (>=3 chars), 4-digit years, multi-digit numbers
    tokens = set(re.findall(r'\b[A-Z][a-zA-Z]{2,}\b|\b\d{4}\b|\b\d{2,}\b', answer))
    # Remove common stop-cap words
    stop = {'The', 'This', 'These', 'That', 'When', 'Where', 'What', 'Which', 'Who', 'How', 'There', 'They', 'Answer'}
    tokens = {t for t in tokens if t not in stop}
    if not tokens:
        return {'flag': 'NO_CHECK', 'coverage': 0.5, 'detail': 'No verifiable terms in answer.'}
    grounded = sum(1 for t in tokens if t.lower() in context_text)
    coverage = grounded / len(tokens)
    if coverage < 0.4:
        flag = 'LIKELY HALLUCINATION'
    elif coverage < 0.7:
        flag = 'PARTIALLY GROUNDED'
    else:
        flag = 'GROUNDED'
    return {'flag': flag, 'coverage': coverage,
            'detail': f'{grounded}/{len(tokens)} key terms found in retrieved context ({coverage:.0%}).'}

def citation_check(answer, chunks):
    """Did the model cite sources matching retrieved chunks?"""
    sources = {c['source'] for c in chunks}
    cited = set(re.findall(r'\[([^\]]+)\]', answer))
    valid = cited & sources
    if not cited:
        return {'cited': False, 'valid_count': 0, 'detail': 'No citations in answer.'}
    return {'cited': True, 'valid_count': len(valid),
            'detail': f'Cited {len(cited)} source(s); {len(valid)} match retrieved chunks: {sorted(valid)}'}

## 19. Pro Dashboard - LLM-as-judge, live metrics, citation panel, retrieval-score bars

## 20. Pipeline trace + embedding viz + 6-metric scorecard (helpers)

In [21]:
import numpy as np
import plotly.graph_objects as go

# --- Pure-numpy 2D PCA (no sklearn dependency) ---
def pca_2d(X):
    X = np.asarray(X, dtype=np.float64)
    X = X - X.mean(axis=0, keepdims=True)
    U, S, Vt = np.linalg.svd(X, full_matrices=False)
    return X @ Vt[:2].T

# --- Trace function: returns ALL retrieval stages for visualization ---
def rag_improved_trace(question, fetch_k=10, final_k=3):
    """Same retrieval pipeline as rag_improved, but returns intermediate stages."""
    q_emb = embed_improved.encode([question]).tolist()
    dense = improved_col.query(query_embeddings=q_emb, n_results=fetch_k)
    dense_hits = []
    for d, m, i, dist in zip(dense['documents'][0], dense['metadatas'][0],
                             dense['ids'][0], dense['distances'][0]):
        dense_hits.append({'text': d, 'source': m['source'], 'id': i,
                           'score': float(1.0 - dist)})

    bm25_scores = bm25.get_scores(tokenize(question))
    top_idx = np.argsort(bm25_scores)[::-1][:fetch_k]
    bm25_hits = []
    for i in top_idx:
        if bm25_scores[i] > 0:
            bm25_hits.append({'text': improved_chunks[i]['text'],
                              'source': improved_chunks[i]['source'],
                              'id': f'improved_{i}',
                              'score': float(bm25_scores[i])})

    seen, fused = set(), []
    for h in dense_hits + bm25_hits:
        if h['id'] not in seen:
            seen.add(h['id'])
            fused.append(h)

    if fused:
        pairs = [[question, h['text']] for h in fused]
        rerank_scores = reranker.predict(pairs)
        ranked = sorted(zip(rerank_scores, fused), key=lambda x: -x[0])
        reranked = [{'text': h['text'], 'source': h['source'], 'id': h['id'],
                     'score': float(s)} for s, h in ranked]
    else:
        reranked = []

    return {'dense': dense_hits, 'bm25': bm25_hits, 'fused': fused,
            'reranked': reranked, 'final': reranked[:final_k]}

# --- Embedding space PCA visualization ---
def embedding_space_plot(question, chunks):
    if not chunks:
        fig = go.Figure()
        fig.add_annotation(text='No chunks to plot', showarrow=False,
                           xref='paper', yref='paper', x=0.5, y=0.5)
        return fig
    q_emb = embed_improved.encode([question])
    chunk_embs = embed_improved.encode([c['text'] for c in chunks])
    all_embs = np.vstack([q_emb, chunk_embs])
    if all_embs.shape[0] < 3:
        coords = np.zeros((all_embs.shape[0], 2))
    else:
        coords = pca_2d(all_embs)

    sizes = [20 + 15 * (c.get('score', 0.5)) for c in chunks]
    short_labels = [c['source'][:18] for c in chunks]
    hover = [f"{c['source'][:25]}<br>score: {c.get('score', 0):.2f}" for c in chunks]

    fig = go.Figure()
    for i in range(len(chunks)):
        fig.add_trace(go.Scatter(
            x=[coords[0, 0], coords[i+1, 0]], y=[coords[0, 1], coords[i+1, 1]],
            mode='lines', line=dict(color='rgba(148,163,184,0.4)', width=1),
            showlegend=False, hoverinfo='skip'))
    fig.add_trace(go.Scatter(
        x=coords[1:, 0], y=coords[1:, 1], mode='markers+text',
        marker=dict(size=sizes, color='#f59e0b', line=dict(color='#92400e', width=1)),
        text=short_labels, textposition='top center', textfont=dict(size=10),
        hovertext=hover, hoverinfo='text', name='Retrieved chunks'))
    fig.add_trace(go.Scatter(
        x=[coords[0, 0]], y=[coords[0, 1]], mode='markers+text',
        marker=dict(size=24, color='#1e40af', symbol='star',
                    line=dict(color='#1e3a8a', width=2)),
        text=['QUERY'], textposition='top center',
        textfont=dict(size=12, color='#1e3a8a'), name='Query'))
    fig.update_layout(height=420, showlegend=True,
        plot_bgcolor='#fafaf9', paper_bgcolor='white',
        margin=dict(l=20, r=20, t=30, b=20),
        title='Embedding space (PCA-2D projection)',
        xaxis=dict(showgrid=True, gridcolor='#e7e5e4', zeroline=False, title=''),
        yaxis=dict(showgrid=True, gridcolor='#e7e5e4', zeroline=False, title=''))
    return fig

# --- 6-metric scorecard ---
def compute_metrics_from_csv(csv_path='/kaggle/working/eval_results.csv'):
    import os
    if not os.path.exists(csv_path):
        return None
    df = pd.read_csv(csv_path)
    def metrics(answer_col, sources_col):
        n = len(df)
        abstain = df[answer_col].str.lower().str.contains("don.t have enough information", na=False).sum()
        cited = df[answer_col].str.contains(r'\[.+?\]', regex=True, na=False).sum()
        halluc = 0
        for _, row in df.iterrows():
            ans = str(row[answer_col]) or ''
            srcs = str(row[sources_col]) or ''
            al = ans.lower()
            if "don.t have enough information" in al or "do not have enough" in al:
                continue
            tokens = set(re.findall(r'\b[A-Z][a-zA-Z]{2,}\b|\b\d{4}\b', ans))
            tokens -= {'The','This','These','That','When','What','Which','Who','How','There','They','Answer'}
            if not tokens:
                continue
            srcs_l = srcs.lower()
            grounded = sum(1 for t in tokens if t.lower() in srcs_l or t.lower() in ans.lower())
            cov = grounded / max(1, len(tokens))
            if cov < 0.4:
                halluc += 1
        hit_rate = ((df[sources_col].fillna('').str.len() > 0).sum()) / n
        return {'hit_rate': hit_rate, 'abstention': abstain / n,
                'citation_rate': cited / n, 'hallucination_rate': halluc / n}
    return {'baseline': metrics('baseline_answer', 'baseline_sources'),
            'improved': metrics('improved_answer', 'improved_sources'),
            'n': len(df)}

print('Helpers loaded: rag_improved_trace, embedding_space_plot, compute_metrics_from_csv')

Helpers loaded: rag_improved_trace, embedding_space_plot, compute_metrics_from_csv


In [23]:
import gradio as gr

# ---------- LLM-as-judge (uses the same model to grade groundedness) ----------
def llm_judge(question, answer, chunks):
    if not answer or not chunks:
        return {'verdict': 'NO_CHECK', 'reason': 'No content to evaluate.'}
    al = answer.lower()
    if "don\'t have enough information" in al or "do not have enough information" in al:
        return {'verdict': 'REFUSED', 'reason': 'Model refused.'}
    context_text = "\n\n".join(f"[{c['source']}] {c['text']}" for c in chunks[:3])
    judge_prompt = (
        "You are a strict evaluator. Decide if the ANSWER is fully supported by the CONTEXT.\n\n"
        f"QUESTION: {question}\n\nCONTEXT:\n{context_text}\n\nANSWER: {answer}\n\n"
        "Reply on ONE line: Verdict: SUPPORTED | PARTIAL | NOT_SUPPORTED || Reason: <one sentence>"
    )
    try:
        out = generate(judge_prompt, max_new_tokens=80)
    except Exception as e:
        return {'verdict': 'ERROR', 'reason': str(e)[:120]}
    verdict = 'UNKNOWN'
    for v in ['SUPPORTED', 'NOT_SUPPORTED', 'PARTIAL']:
        if v in out.upper():
            verdict = v
            break
    reason = out.split('Reason:')[-1].strip()[:200] if 'Reason:' in out else out[:200]
    return {'verdict': verdict, 'reason': reason}

# ---------- Session metrics ----------
SESSION = {'asked': 0,
    'baseline': {'GROUNDED': 0, 'PARTIALLY GROUNDED': 0, 'LIKELY HALLUCINATION': 0, 'REFUSED': 0, 'NO_CHECK': 0},
    'improved': {'GROUNDED': 0, 'PARTIALLY GROUNDED': 0, 'LIKELY HALLUCINATION': 0, 'REFUSED': 0, 'NO_CHECK': 0},
    'history': []}

def badge_html(flag):
    palette = {'GROUNDED': '#16a34a', 'PARTIALLY GROUNDED': '#d97706',
               'LIKELY HALLUCINATION': '#dc2626', 'REFUSED': '#0ea5e9',
               'NO_CHECK': '#6b7280', 'SUPPORTED': '#16a34a',
               'NOT_SUPPORTED': '#dc2626', 'PARTIAL': '#d97706',
               'UNKNOWN': '#6b7280', 'ERROR': '#6b7280'}
    c = palette.get(flag, '#6b7280')
    return f'<span style="background:{c};color:white;padding:5px 12px;border-radius:8px;font-weight:600;font-size:13px;">{flag}</span>'

def chunks_html(chunks):
    if not chunks:
        return '<em style="color:#999">No chunks retrieved.</em>'
    out = ['<div style="display:flex;flex-direction:column;gap:8px;">']
    for i, c in enumerate(chunks[:3], 1):
        text = c['text'][:280] + ('...' if len(c['text']) > 280 else '')
        score_html = ''
        if 'score' in c:
            score_html = f'<span style="font-size:11px;color:#92400e;">score {c["score"]:.2f}</span>'
        out.append(f'<div style="border:1px solid #e2e8f0;border-radius:8px;padding:10px;background:#f8fafc;">'
                   f'<div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:6px;">'
                   f'<span style="background:#3b82f6;color:white;padding:2px 8px;border-radius:6px;font-size:11px;font-weight:600;">#{i}</span>'
                   f'<span style="font-size:12px;color:#475569;font-weight:600;">{c["source"]}</span>{score_html}</div>'
                   f'<div style="font-size:12px;color:#334155;line-height:1.5;">{text}</div></div>')
    out.append('</div>')
    return '\n'.join(out)

def scoreboard_html():
    s = SESSION
    b_good = s['baseline']['GROUNDED'] + s['baseline']['REFUSED']
    i_good = s['improved']['GROUNDED'] + s['improved']['REFUSED']
    b_bad = s['baseline']['LIKELY HALLUCINATION']
    i_bad = s['improved']['LIKELY HALLUCINATION']
    def stat(label, val, color='#0f172a'):
        return (f'<div style="text-align:center;padding:8px 16px;">'
                f'<div style="font-size:24px;font-weight:700;color:{color};">{val}</div>'
                f'<div style="font-size:11px;color:#64748b;text-transform:uppercase;letter-spacing:.5px;">{label}</div></div>')
    return ('<div style="display:flex;justify-content:space-around;background:linear-gradient(90deg,#eff6ff,#f0f9ff);'
            'border:1px solid #bfdbfe;border-radius:12px;padding:12px;margin:8px 0;">'
            + stat('Questions Asked', s['asked'])
            + stat('Baseline Grounded+Refused', b_good, '#16a34a')
            + stat('Improved Grounded+Refused', i_good, '#16a34a')
            + stat('Baseline Hallucinations', b_bad, '#dc2626')
            + stat('Improved Hallucinations', i_bad, '#dc2626')
            + '</div>')

def metrics_chart():
    cats = ['GROUNDED', 'PARTIALLY GROUNDED', 'LIKELY HALLUCINATION', 'REFUSED']
    fig = go.Figure()
    fig.add_trace(go.Bar(name='Baseline', x=cats, y=[SESSION['baseline'][c] for c in cats], marker_color='#94a3b8'))
    fig.add_trace(go.Bar(name='Improved', x=cats, y=[SESSION['improved'][c] for c in cats], marker_color='#3b82f6'))
    fig.update_layout(barmode='group', height=320, margin=dict(l=20,r=20,t=20,b=40),
                      legend=dict(orientation='h', y=1.15), plot_bgcolor='white')
    return fig

def ask_compare(question, use_judge):
    if not question or not question.strip():
        return ("", "", "", "", "", "", scoreboard_html(), metrics_chart())
    try:
        ans_b, chunks_b = rag_baseline(question)
    except Exception as e:
        ans_b, chunks_b = f"ERROR: {e}", []
    try:
        ans_i, chunks_i = rag_improved(question)
    except Exception as e:
        ans_i, chunks_i = f"ERROR: {e}", []
    hb = hallucination_heuristic(ans_b, chunks_b)
    hi = hallucination_heuristic(ans_i, chunks_i)
    cb = citation_check(ans_b, chunks_b)
    ci = citation_check(ans_i, chunks_i)
    jb_html = ji_html = ''
    if use_judge:
        jb = llm_judge(question, ans_b, chunks_b)
        ji = llm_judge(question, ans_i, chunks_i)
        jb_html = f'<div style="margin-top:8px;"><strong>LLM-as-judge:</strong> {badge_html(jb["verdict"])} <span style="font-size:12px;color:#475569;">{jb["reason"]}</span></div>'
        ji_html = f'<div style="margin-top:8px;"><strong>LLM-as-judge:</strong> {badge_html(ji["verdict"])} <span style="font-size:12px;color:#475569;">{ji["reason"]}</span></div>'
    SESSION['asked'] += 1
    SESSION['baseline'][hb['flag']] = SESSION['baseline'].get(hb['flag'], 0) + 1
    SESSION['improved'][hi['flag']] = SESSION['improved'].get(hi['flag'], 0) + 1
    flag_b_html = (f'<div style="display:flex;flex-direction:column;gap:6px;">'
                   f'<div>{badge_html(hb["flag"])} <span style="font-size:12px;color:#475569;">{hb["detail"]}</span></div>'
                   f'<div style="font-size:12px;color:#475569;"><strong>Citations:</strong> {cb["detail"]}</div>{jb_html}</div>')
    flag_i_html = (f'<div style="display:flex;flex-direction:column;gap:6px;">'
                   f'<div>{badge_html(hi["flag"])} <span style="font-size:12px;color:#475569;">{hi["detail"]}</span></div>'
                   f'<div style="font-size:12px;color:#475569;"><strong>Citations:</strong> {ci["detail"]}</div>{ji_html}</div>')
    return (ans_b, flag_b_html, chunks_html(chunks_b),
            ans_i, flag_i_html, chunks_html(chunks_i),
            scoreboard_html(), metrics_chart())

def reset_session():
    SESSION['asked'] = 0
    for k in SESSION['baseline']:
        SESSION['baseline'][k] = 0; SESSION['improved'][k] = 0
    SESSION['history'].clear()
    return scoreboard_html(), metrics_chart()

# ---------- Pipeline trace UI ----------
def trace_pipeline(question):
    """Run the trace and return: 4 stage HTMLs + embedding plot."""
    if not question or not question.strip():
        empty = '<em style="color:#999">Enter a question</em>'
        return empty, empty, empty, empty, go.Figure()
    trace = rag_improved_trace(question, fetch_k=10, final_k=3)
    def stage_html(stage_name, hits, color):
        if not hits:
            return f'<div style="color:#999"><em>Empty</em></div>'
        rows = []
        for i, h in enumerate(hits[:8], 1):
            preview = h['text'][:140] + ('...' if len(h['text']) > 140 else '')
            rows.append(
                f'<div style="border-left:3px solid {color};background:#f8fafc;padding:8px 12px;margin-bottom:6px;border-radius:0 6px 6px 0;">'
                f'<div style="display:flex;justify-content:space-between;font-size:11px;color:#475569;margin-bottom:4px;">'
                f'<strong>#{i} &middot; {h["source"][:25]}</strong>'
                f'<span style="color:{color};font-weight:600;">{h["score"]:.3f}</span></div>'
                f'<div style="font-size:11px;color:#334155;line-height:1.4;">{preview}</div></div>'
            )
        return '<div>' + ''.join(rows) + '</div>'
    dense_html = stage_html('Dense', trace['dense'], '#3b82f6')
    bm25_html = stage_html('BM25', trace['bm25'], '#f59e0b')
    fused_html = stage_html('Fused', trace['fused'], '#8b5cf6')
    reranked_html = stage_html('Reranked', trace['reranked'], '#16a34a')
    emb_plot = embedding_space_plot(question, trace['final'])
    return dense_html, bm25_html, fused_html, reranked_html, emb_plot

def run_full_eval():
    rows = []
    for i, item in enumerate(EVAL_QUESTIONS, 1):
        try: ans_b, c_b = rag_baseline(item['q'])
        except Exception as e: ans_b, c_b = f'ERROR: {e}', []
        try: ans_i, c_i = rag_improved(item['q'])
        except Exception as e: ans_i, c_i = f'ERROR: {e}', []
        hb = hallucination_heuristic(ans_b, c_b)
        hi = hallucination_heuristic(ans_i, c_i)
        rows.append([i, item['type'], item['q'][:60],
                     ans_b[:100], hb['flag'],
                     ans_i[:100], hi['flag']])
    return rows

# ---------- Six-metric scorecard ----------
def render_scorecard():
    m = compute_metrics_from_csv('/kaggle/working/eval_results.csv')
    if m is None:
        return '<em style="color:#dc2626;">Run the eval first (cell 14) so eval_results.csv exists, then click Refresh.</em>'
    def card(label, b_pct, i_pct, lower_better=False):
        b_color = '#dc2626' if (lower_better and b_pct > i_pct) or (not lower_better and b_pct < i_pct) else '#16a34a'
        i_color = '#dc2626' if (lower_better and i_pct > b_pct) or (not lower_better and i_pct < b_pct) else '#16a34a'
        return (f'<div style="border:1px solid #e2e8f0;border-radius:12px;padding:16px;background:white;">'
                f'<div style="font-size:13px;color:#475569;font-weight:600;margin-bottom:12px;text-transform:uppercase;letter-spacing:.5px;">{label}</div>'
                f'<div style="display:flex;flex-direction:column;gap:8px;">'
                f'<div style="display:flex;align-items:center;gap:10px;"><span style="width:75px;font-size:12px;color:#64748b;">baseline</span>'
                f'<div style="flex:1;height:10px;background:#f1f5f9;border-radius:5px;overflow:hidden;"><div style="width:{int(b_pct*100)}%;height:100%;background:#94a3b8;"></div></div>'
                f'<span style="width:50px;text-align:right;font-weight:700;color:{b_color};">{int(b_pct*100)}%</span></div>'
                f'<div style="display:flex;align-items:center;gap:10px;"><span style="width:75px;font-size:12px;color:#64748b;">improved</span>'
                f'<div style="flex:1;height:10px;background:#f1f5f9;border-radius:5px;overflow:hidden;"><div style="width:{int(i_pct*100)}%;height:100%;background:#3b82f6;"></div></div>'
                f'<span style="width:50px;text-align:right;font-weight:700;color:{i_color};">{int(i_pct*100)}%</span></div>'
                f'</div></div>')
    b, i = m['baseline'], m['improved']
    cards = [
        card('Retrieval hit-rate', b['hit_rate'], i['hit_rate']),
        card('Citation rate', b['citation_rate'], i['citation_rate']),
        card('Abstention on unanswerable', b['abstention'], i['abstention']),
        card('Hallucination rate', b['hallucination_rate'], i['hallucination_rate'], lower_better=True),
    ]
    header = f'<div style="margin-bottom:12px;color:#475569;"><strong>Computed from /kaggle/working/eval_results.csv</strong> &middot; n={m["n"]} questions</div>'
    return header + '<div style="display:grid;grid-template-columns:1fr 1fr;gap:14px;">' + ''.join(cards) + '</div>'

# ---------- UI ----------
SAMPLE = ['When was the James Webb Space Telescope launched?',
          'What is the largest moon of Jupiter?',
          'Compare the missions of Curiosity and Perseverance rovers.',
          'Which planets have ring systems?',
          'Tell me about Mars.', 'What is a giant?', 'How big is it?']
TRAPS = ['What did Voyager 3 discover during its mission?',
         'Summarize the successful Pluto landing mission of 2020.',
         'Who was the first Indian astronaut to walk on Mars?',
         "What was Hubble's first photograph of a black hole in 2024?",
         'What did Cassini-Huygens find on the rings of Mercury?']

custom_css = """
.gradio-container {font-family: -apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,Helvetica,Arial,sans-serif !important;}
h1,h2,h3 {color:#0f172a;}
.pipeline-tag {display:inline-block;padding:3px 10px;border-radius:6px;font-size:12px;font-weight:600;}
.tag-baseline {background:#f1f5f9;color:#475569;}
.tag-improved {background:#dbeafe;color:#1e40af;}
.stage-badge {display:inline-block;padding:3px 10px;border-radius:6px;font-size:12px;font-weight:600;color:white;margin-right:6px;}
"""

with gr.Blocks(title='Astronomy RAG - Pro Dashboard', theme=gr.themes.Soft(primary_hue='blue'), css=custom_css) as demo:
    gr.Markdown('# Astronomy RAG - Glass-box Comparison Dashboard')
    gr.Markdown('Same question, both pipelines, every step visible: retrieved chunks, hallucination heuristic, citation check, LLM-as-judge, retrieval-stage trace, and 2D embedding space.')
    scoreboard = gr.HTML(scoreboard_html())

    with gr.Tab('Ask and Compare'):
        with gr.Row():
            with gr.Column(scale=4):
                q_in = gr.Textbox(label='Question', placeholder='Ask anything about astronomy...', lines=2)
            with gr.Column(scale=1):
                ask_btn = gr.Button('Ask Both Pipelines', variant='primary', size='lg')
                use_judge = gr.Checkbox(label='Enable LLM-as-judge (+10s)', value=False)
                reset_btn = gr.Button('Reset Session Metrics', size='sm')
        sample_dd = gr.Dropdown(SAMPLE, label='Or pick a sample question', interactive=True)
        sample_dd.change(lambda x: x, inputs=sample_dd, outputs=q_in)
        with gr.Row():
            with gr.Column():
                gr.Markdown('### <span class="pipeline-tag tag-baseline">BASELINE</span> &nbsp; Fixed chunks | MiniLM | dense top-3 | plain prompt')
                b_ans = gr.Textbox(label='Answer', lines=5)
                b_flag = gr.HTML()
                gr.Markdown('**Retrieved chunks**')
                b_chunks = gr.HTML()
            with gr.Column():
                gr.Markdown('### <span class="pipeline-tag tag-improved">IMPROVED</span> &nbsp; Paragraph chunks | BGE | hybrid + rerank | citation prompt')
                i_ans = gr.Textbox(label='Answer', lines=5)
                i_flag = gr.HTML()
                gr.Markdown('**Retrieved chunks (after cross-encoder rerank)**')
                i_chunks = gr.HTML()
        with gr.Accordion('Session metrics chart', open=False):
            chart = gr.Plot(metrics_chart())
        ask_btn.click(ask_compare, inputs=[q_in, use_judge],
                      outputs=[b_ans, b_flag, b_chunks, i_ans, i_flag, i_chunks, scoreboard, chart])
        q_in.submit(ask_compare, inputs=[q_in, use_judge],
                    outputs=[b_ans, b_flag, b_chunks, i_ans, i_flag, i_chunks, scoreboard, chart])
        reset_btn.click(reset_session, outputs=[scoreboard, chart])

    with gr.Tab('Pipeline Trace'):
        gr.Markdown('## Trace a single query through every retrieval stage')
        gr.Markdown('Run a question through the improved retriever and watch the chunks flow Dense -> BM25 -> Fused -> Reranked. The embedding-space plot shows where the final-3 chunks sit relative to the query in 2D PCA space.')
        with gr.Row():
            trace_q = gr.Textbox(label='Question', placeholder='Try: "Compare Curiosity and Perseverance rovers"', scale=4)
            trace_btn = gr.Button('Trace pipeline', variant='primary', scale=1)
        with gr.Row():
            with gr.Column():
                gr.Markdown('### <span class="stage-badge" style="background:#3b82f6;">1. DENSE</span> Vector similarity (BGE cosine)')
                dense_panel = gr.HTML()
            with gr.Column():
                gr.Markdown('### <span class="stage-badge" style="background:#f59e0b;">2. SPARSE</span> BM25 keyword match')
                bm25_panel = gr.HTML()
        with gr.Row():
            with gr.Column():
                gr.Markdown('### <span class="stage-badge" style="background:#8b5cf6;">3. FUSED</span> Union dedupe')
                fused_panel = gr.HTML()
            with gr.Column():
                gr.Markdown('### <span class="stage-badge" style="background:#16a34a;">4. RERANKED</span> Cross-encoder scoring')
                rerank_panel = gr.HTML()
        gr.Markdown('### Embedding space - query vs final retrieved chunks (PCA 2D)')
        emb_plot = gr.Plot()
        trace_btn.click(trace_pipeline, inputs=trace_q,
                       outputs=[dense_panel, bm25_panel, fused_panel, rerank_panel, emb_plot])

    with gr.Tab('Failure Lab'):
        gr.Markdown('## False-premise / hallucination trap questions')
        gr.Markdown('Baseline tends to fabricate. Improved should refuse cleanly.')
        with gr.Row():
            trap_dd = gr.Dropdown(TRAPS, label='Pick a trap question', interactive=True, scale=4)
            run_trap = gr.Button('Run', variant='primary', scale=1)
            use_judge_trap = gr.Checkbox(label='Enable LLM-as-judge', value=False)
        with gr.Row():
            with gr.Column():
                gr.Markdown('### BASELINE')
                tb_ans = gr.Textbox(label='Answer', lines=5)
                tb_flag = gr.HTML()
            with gr.Column():
                gr.Markdown('### IMPROVED')
                ti_ans = gr.Textbox(label='Answer', lines=5)
                ti_flag = gr.HTML()
        def run_trap_fn(q, uj):
            r = ask_compare(q, uj)
            return r[0], r[1], r[3], r[4], r[6], r[7]
        run_trap.click(run_trap_fn, inputs=[trap_dd, use_judge_trap],
                       outputs=[tb_ans, tb_flag, ti_ans, ti_flag, scoreboard, chart])

    with gr.Tab('Eval + Metrics Scorecard'):
        gr.Markdown('## Run all 18 questions and view the metric scorecard')
        with gr.Row():
            run_eval_btn = gr.Button('Run 18-question evaluation', variant='primary')
            refresh_metrics_btn = gr.Button('Refresh scorecard from CSV')
        gr.Markdown('### Six-metric scorecard')
        scorecard_html = gr.HTML(render_scorecard())
        gr.Markdown('### Per-question table')
        eval_tbl = gr.Dataframe(
            headers=['#', 'Type', 'Question', 'Baseline Answer', 'B Flag', 'Improved Answer', 'I Flag'],
            label='Full evaluation results', wrap=True,
        )
        def run_and_refresh():
            rows = run_full_eval()
            sc = render_scorecard()
            return rows, sc
        run_eval_btn.click(run_and_refresh, outputs=[eval_tbl, scorecard_html])
        refresh_metrics_btn.click(render_scorecard, outputs=scorecard_html)

    with gr.Tab('How it works'):
        gr.Markdown("""## How this dashboard works

**Side-by-side runs** make each improvement immediately visible. Same question, both pipelines, both sets of retrieved chunks shown.

**Three quality layers**: heuristic hallucination check (lexical groundedness), citation check (regex against retrieved sources), and optional LLM-as-judge (the model evaluates its own answer).

**Pipeline Trace** lets you inspect any query at every retrieval stage with rank-by-rank scores, plus a 2D PCA projection of the embedding space showing where the final retrieved chunks sit relative to the query.

**Eval + Metrics Scorecard** computes Atlas-style metrics from the eval CSV: retrieval hit-rate, citation rate, abstention rate, hallucination rate. Click 'Refresh scorecard' after the eval completes.

**Live session metrics** at the top of the page update as you ask questions. Reset whenever you want.

**Failure Lab** runs preset false-premise questions to demonstrate the hallucination-vs-refusal contrast.
""")

demo.launch(share=True, debug=False)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://007bdbd71d18b25187.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

## 20. Stop the dashboard (optional)

If you need to free the port to relaunch:

In [ ]:
# demo.close()